# conv-stride-downsample — faded example 3: Emulate stride-K downsample with einops reduce

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-stride-downsample`. Running the beacon reports progress on the `CNN: Stride downsample arithmetic` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Stride downsample arithmetic` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-stride-downsample`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-stride-downsample"
DD_SUBTOPIC = "CNN: Stride downsample arithmetic"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A non-overlapping stride-`K` window of size `K` (kernel size equals stride) partitions the signal into disjoint blocks, so a strided conv/pool over it is just a per-block reduction. With einops you reshape `(b c (l k)) -> (b c l)` and reduce over `k`, dropping any ragged tail. This makes the downsample-by-K arithmetic explicit: output length is `L // K`.

## Faded exercise 3

### Faded — strided downsample as an einops block reduce

Implement `block_mean_downsample(x, k)` that downsamples the last axis of `x` (shape `(b, c, l)`) by a factor of `k` using non-overlapping mean pooling. Truncate the signal to a multiple of `k` first, then complete the einops `reduce` call. The test compares against `F.avg_pool1d` with kernel and stride both `k`.

**Fill in:** The einops `reduce` call that averages each non-overlapping block of size `k`: `reduce(x, 'b c (l k) -> b c l', 'mean', k=k)`.

In [ ]:
import torch.nn.functional as F

def block_mean_downsample(x, k):
    l = x.shape[-1]
    l_trunc = (l // k) * k
    x = x[..., :l_trunc]
    out = reduce(x, 'b c (l k) -> b c l', 'mean', k=k)
    return out


def _test():
    t.manual_seed(0)
    for l, k in [(20, 4), (33, 3), (16, 2), (50, 5)]:
        x = t.randn(2, 3, l)
        out = block_mean_downsample(x, k)
        ref = F.avg_pool1d(x, kernel_size=k, stride=k)
        assert out.shape == ref.shape, (l, k, tuple(out.shape), tuple(ref.shape))
        assert out.shape[-1] == l // k
        assert t.allclose(out, ref, atol=1e-5), (l, k)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn.functional as F

def block_mean_downsample(x, k):
    l = x.shape[-1]
    l_trunc = (l // k) * k
    x = x[..., :l_trunc]
    out = reduce(x, 'b c (l k) -> b c l', 'mean', k=k)
    return out
```
</details>